# FINAL HUMAN EXPERT VALIDATION DATASET BUILDER

# Semantic Coverage + Additional Metrics

# IMPORTANT:
# 1. Reads EACH Report_X_uml_pages folder directly.
# 2. DOES NOT use any all_reports_* aggregate folder.
# 3. DOES NOT recompute any metric.
# 4. DOES NOT modify existing experimental results.
# 5. Extracts existing semantic-coverage metric values.
# 6. Extracts existing additional coverage values.
# 7. Preserves exact semantic metric names used in each result.
# 8. Creates an Excel workbook ready for two human experts.

# Expected:
# 4 LLMs x 5 VLMs x 29 reports = 580 rows


# MOUNT GOOGLE DRIVE

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# IMPORTS

In [ ]:
import os
import re
import glob
import json
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side
)
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

# 2. ROOT DIRECTORY


In [ ]:
SEMANTIC_COVERAGE_ROOT = (
    "/content/drive/MyDrive/SEMANTIC_COVERAGE_READY11"
)

OUT_DIR = os.path.join(
    SEMANTIC_COVERAGE_ROOT,
    "Human_Expert_Validation"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

OUT_XLSX = os.path.join(
    OUT_DIR,
    "Human_Validation_All_Semantic_Additional_Combinations.xlsx"
)

In [ ]:
# EXACT LLM FOLDER NAMES


LLM_FOLDERS = {
    "mistral 7b":
        "Mistral-7B-Instruct-v0.3",

    "mistral 14b":
        "Ministral-3-14B-Instruct-2512",

    "qwen 7b":
        "Qwen2.5-7B-Instruct",

    "qwen 14b":
        "Qwen2.5-14B-Instruct",
}


# EXACT VLM FOLDER NAMES

VLM_FOLDERS = {
    "gemma 4b":
        "Gemma-3-4B",

    "gemma 12b":
        "Gemma-3-12B",

    "llama":
        "Llama-3.2-11B-Vision",

    "llava":
        "LLaVA-1.6-13B",

    "qwen":
        "Qwen2.5-VL-7B",
}


# EXACT 29 REPORTS

EXPECTED_REPORTS = [
    "Report_1_uml_pages",
    "Report_2_uml_pages",
    "Report_3_uml_pages",
    "Report_4_uml_pages",
    "Report_5_uml_pages",
    "Report_6_uml_pages",
    "Report_7_uml_pages",
    "Report_8_uml_pages",
    "Report_9_uml_pages",
    "Report_10_uml_pages",
    "Report_12_uml_pages",
    "Report_13_uml_pages",
    "Report_14_uml_pages",
    "Report_15_uml_pages",
    "Report_16_uml_pages",
    "Report_19_uml_pages",
    "Report_21_uml_pages",
    "Report_22_uml_pages",
    "Report_23_uml_pages",
    "Report_24_uml_pages",
    "Report_25_uml_pages",
    "Report_26_uml_pages",
    "Report_27_uml_pages",
    "Report_28_uml_pages",
    "Report_29_uml_pages",
    "Report_30_uml_pages",
    "Report_31_uml_pages",
    "Report_32_uml_pages",
    "Report_33_uml_pages",
]

assert len(EXPECTED_REPORTS) == 29


# BASIC HELPERS


def norm(x):
    """
    Normalize names for flexible matching.
    """
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(x).lower()
    )


def report_number(x):

    m = re.search(
        r"(\d+)",
        str(x)
    )

    return (
        int(m.group(1))
        if m
        else 999999
    )


def safe_float(x):

    try:

        if pd.isna(x):
            return np.nan

        return float(x)

    except:
        return np.nan


def first_existing_column(
    df,
    candidates
):

    if df is None:
        return None

    mapping = {
        norm(c): c
        for c in df.columns
    }

    for candidate in candidates:

        candidate_norm = norm(
            candidate
        )

        if candidate_norm in mapping:

            return mapping[
                candidate_norm
            ]

    return None


def first_numeric_value(
    df,
    candidates
):

    col = first_existing_column(
        df,
        candidates
    )

    if col is None:
        return np.nan

    if len(df) == 0:
        return np.nan

    return safe_float(
        df.iloc[0][col]
    )


# READ CSV / EXCEL / JSON


def read_result_file(path):

    if path is None:
        return None

    try:

        lower = path.lower()

        if lower.endswith(".csv"):

            return pd.read_csv(
                path
            )

        elif (
            lower.endswith(".xlsx")
            or lower.endswith(".xls")
        ):

            return pd.read_excel(
                path
            )

        elif lower.endswith(".json"):

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:

                obj = json.load(f)

            if isinstance(
                obj,
                dict
            ):
                return pd.DataFrame(
                    [obj]
                )

            if isinstance(
                obj,
                list
            ):
                return pd.DataFrame(
                    obj
                )

    except Exception as e:

        print(
            "Could not read:",
            path
        )

        print(
            "Reason:",
            e
        )

    return None



# 8. FIND FILE INSIDE ONE REPORT ONLY
# This is the important correction.
# Search starts at:
# LLM / VLM / Report_X_uml_pages
# It NEVER searches all_reports_* folders.

def find_report_result(
    report_dir,
    filenames
):

    found = []

    for filename in filenames:

        matches = glob.glob(
            os.path.join(
                report_dir,
                "**",
                filename
            ),
            recursive=True
        )

        for path in matches:

            # Protection:
            # never accept anything under
            # an all_reports aggregate folder.
            path_norm = norm(path)

            if "allreports" in path_norm:
                continue

            found.append(
                path
            )

    if not found:
        return None

    # Prefer CSV, then XLSX, then JSON
    def preference(path):

        lower = path.lower()

        if lower.endswith(".csv"):
            type_order = 0

        elif lower.endswith(".xlsx"):
            type_order = 1

        elif lower.endswith(".xls"):
            type_order = 2

        elif lower.endswith(".json"):
            type_order = 3

        else:
            type_order = 4

        return (
            type_order,
            len(path),
            path
        )

    found = sorted(
        set(found),
        key=preference
    )

    return found[0]


# FIND SOURCE ARTIFACTS
#
# Useful for expert inspection.


def find_source_artifacts(
    report_dir
):

    # UML description TXT

    txt_candidates = glob.glob(
        os.path.join(
            report_dir,
            "*.txt"
        )
    )

    uml_txt = (
        sorted(txt_candidates)[0]
        if txt_candidates
        else ""
    )

    # Original generated testcase CSV
    # Only root-level CSVs, not result CSVs

    csv_candidates = glob.glob(
        os.path.join(
            report_dir,
            "*.csv"
        )
    )

    testcase_csv = ""

    for path in sorted(
        csv_candidates
    ):

        filename = os.path.basename(
            path
        ).lower()

        if (
            "test" in filename
            or "case" in filename
        ):

            testcase_csv = path
            break

    if (
        not testcase_csv
        and csv_candidates
    ):

        testcase_csv = sorted(
            csv_candidates
        )[0]

    return (
        uml_txt,
        testcase_csv
    )


# 10. PARSE SEMANTIC COVERAGE
#
# Supports BOTH semantic naming structures:
#
# Newer:
#   EntityCoverage
#   FlowCoverage
#   ConditionalCoverage
#   SemanticCoverage
#
# Earlier:
#   ConceptCoverage
#   FlowCoverage
#   ValidationCoverage
#   SemanticCoverage
#
# Values are READ ONLY.
# No semantic value is recalculated.

def parse_semantic_coverage(
    report_dir
):

    semantic_path = find_report_result(
        report_dir,
        [
            "semantic_coverage_summary.csv",
            "semantic_coverage_summary.xlsx",
            "semantic_coverage_summary.xls",
            "semantic_coverage_summary.json",
        ]
    )

    empty_result = {
        "Semantic_Element_Metric_Name":
            "",

        "Automated_Element_Coverage":
            np.nan,

        "Semantic_Flow_Metric_Name":
            "",

        "Automated_Flow_Coverage":
            np.nan,

        "Semantic_Conditional_Metric_Name":
            "",

        "Automated_Conditional_Coverage":
            np.nan,

        "Semantic_Overall_Metric_Name":
            "",

        "Automated_ELSC":
            np.nan,
    }

    if semantic_path is None:

        return (
            empty_result,
            None,
            []
        )

    df = read_result_file(
        semantic_path
    )

    if (
        df is None
        or len(df) == 0
    ):

        return (
            empty_result,
            semantic_path,
            []
        )

    raw_metric_rows = []


    # FORMAT A:
    #
    # metric                value
    # EntityCoverage        ...
    # FlowCoverage          ...
    # ConditionalCoverage   ...
    # SemanticCoverage      ...


    metric_col = first_existing_column(
        df,
        [
            "metric"
        ]
    )

    value_col = first_existing_column(
        df,
        [
            "value"
        ]
    )

    if (
        metric_col is not None
        and value_col is not None
    ):

        metric_map = {}

        original_name_map = {}

        for _, row in df.iterrows():

            original_metric = str(
                row[metric_col]
            )

            normalized_metric = norm(
                original_metric
            )

            value = safe_float(
                row[value_col]
            )

            metric_map[
                normalized_metric
            ] = value

            original_name_map[
                normalized_metric
            ] = original_metric

            raw_metric_rows.append({
                "Metric":
                    original_metric,

                "Value":
                    value,
            })


        # Element / Entity / Concept

        element_candidates = [
            "entitycoverage",
            "conceptcoverage",
        ]

        element_name = ""
        element_value = np.nan

        for candidate in element_candidates:

            if candidate in metric_map:

                element_name = (
                    original_name_map[
                        candidate
                    ]
                )

                element_value = (
                    metric_map[
                        candidate
                    ]
                )

                break

        # Flow

        flow_name = ""
        flow_value = np.nan

        if "flowcoverage" in metric_map:

            flow_name = (
                original_name_map[
                    "flowcoverage"
                ]
            )

            flow_value = (
                metric_map[
                    "flowcoverage"
                ]
            )

        # Conditional / Validation

        conditional_candidates = [
            "conditionalcoverage",
            "validationcoverage",
        ]

        conditional_name = ""
        conditional_value = np.nan

        for candidate in conditional_candidates:

            if candidate in metric_map:

                conditional_name = (
                    original_name_map[
                        candidate
                    ]
                )

                conditional_value = (
                    metric_map[
                        candidate
                    ]
                )

                break

        # Overall semantic coverage

        overall_name = ""
        overall_value = np.nan

        for candidate in [
            "semanticcoverage",
            "overallsemanticcoverage",
        ]:

            if candidate in metric_map:

                overall_name = (
                    original_name_map[
                        candidate
                    ]
                )

                overall_value = (
                    metric_map[
                        candidate
                    ]
                )

                break

        result = {
            "Semantic_Element_Metric_Name":
                element_name,

            "Automated_Element_Coverage":
                element_value,

            "Semantic_Flow_Metric_Name":
                flow_name,

            "Automated_Flow_Coverage":
                flow_value,

            "Semantic_Conditional_Metric_Name":
                conditional_name,

            "Automated_Conditional_Coverage":
                conditional_value,

            "Semantic_Overall_Metric_Name":
                overall_name,

            "Automated_ELSC":
                overall_value,
        }

        return (
            result,
            semantic_path,
            raw_metric_rows
        )

    # FORMAT B:
    #
    # One-row wide summary:
    #
    # report_folder
    # entity_coverage
    # flow_coverage
    # conditional_coverage
    # semantic_coverage
    #
    # OR concept / validation naming

    row = df.iloc[
        0:1
    ].copy()

    # Entity / Concept

    element_col = first_existing_column(
        row,
        [
            "entity_coverage",
            "concept_coverage",
            "EntityCoverage",
            "ConceptCoverage",
        ]
    )

    element_value = (
        safe_float(
            row.iloc[0][
                element_col
            ]
        )
        if element_col
        else np.nan
    )

    # Flow

    flow_col = first_existing_column(
        row,
        [
            "flow_coverage",
            "FlowCoverage",
        ]
    )

    flow_value = (
        safe_float(
            row.iloc[0][
                flow_col
            ]
        )
        if flow_col
        else np.nan
    )

    # Conditional / Validation

    conditional_col = (
        first_existing_column(
            row,
            [
                "conditional_coverage",
                "validation_coverage",
                "ConditionalCoverage",
                "ValidationCoverage",
            ]
        )
    )

    conditional_value = (
        safe_float(
            row.iloc[0][
                conditional_col
            ]
        )
        if conditional_col
        else np.nan
    )

    # Overall semantic coverage

    overall_col = first_existing_column(
        row,
        [
            "semantic_coverage",
            "overall_semantic_coverage",
            "SemanticCoverage",
        ]
    )

    overall_value = (
        safe_float(
            row.iloc[0][
                overall_col
            ]
        )
        if overall_col
        else np.nan
    )

    result = {
        "Semantic_Element_Metric_Name":
            element_col or "",

        "Automated_Element_Coverage":
            element_value,

        "Semantic_Flow_Metric_Name":
            flow_col or "",

        "Automated_Flow_Coverage":
            flow_value,

        "Semantic_Conditional_Metric_Name":
            conditional_col or "",

        "Automated_Conditional_Coverage":
            conditional_value,

        "Semantic_Overall_Metric_Name":
            overall_col or "",

        "Automated_ELSC":
            overall_value,
    }

    # Keep every numeric field for raw traceability
    for col in df.columns:

        value = df.iloc[0][col]

        if isinstance(
            value,
            (int, float, np.integer, np.floating)
        ):

            raw_metric_rows.append({
                "Metric":
                    col,

                "Value":
                    safe_float(value),
            })

    return (
        result,
        semantic_path,
        raw_metric_rows
    )


# THRESHOLD COLUMN FINDER

def threshold_value(
    df,
    prefix,
    threshold
):

    if (
        df is None
        or len(df) == 0
    ):
        return np.nan

    prefix_norm = norm(
        prefix
    )

    threshold_digits = str(
        threshold
    ).replace(
        ".",
        ""
    )

    for col in df.columns:

        col_norm = norm(
            col
        )

        if (
            prefix_norm in col_norm
            and threshold_digits in col_norm
        ):

            return safe_float(
                df.iloc[0][col]
            )

    return np.nan


# 12. PARSE ADDITIONAL METRICS
#
# Extracts:
#
# COC
# COC@0.65 / 0.70 / 0.75
#
# CBTC
# CBTC@0.65 / 0.70 / 0.75
#
# ERPC
#
# Again: no metric is recalculated.

def parse_additional_metrics(
    report_dir
):

    additional_path = find_report_result(
        report_dir,
        [
            "additional_coverage_metrics_summary.csv",
            "additional_coverage_metrics_summary.xlsx",
            "additional_coverage_metrics_summary.xls",
            "additional_coverage_metrics_summary.json",
        ]
    )

    empty = {
        "Num_UML_Test_Objectives":
            np.nan,

        "Automated_COC":
            np.nan,

        "COC_0.65":
            np.nan,

        "COC_0.70":
            np.nan,

        "COC_0.75":
            np.nan,

        "Num_Condition_Bound_Objectives":
            np.nan,

        "CBTC_Applicable":
            "",

        "Automated_CBTC":
            np.nan,

        "CBTC_0.65":
            np.nan,

        "CBTC_0.70":
            np.nan,

        "CBTC_0.75":
            np.nan,

        "Automated_ERPC":
            np.nan,

        "Num_Testcases_With_Expected_Result":
            np.nan,

        "Num_Testcases":
            np.nan,
    }

    if additional_path is None:

        return (
            empty,
            None,
            None
        )

    df = read_result_file(
        additional_path
    )

    if (
        df is None
        or len(df) == 0
    ):

        return (
            empty,
            additional_path,
            df
        )

    row = df.iloc[
        0:1
    ].copy()

    # Number of UML test objectives

    num_objectives = first_numeric_value(
        row,
        [
            "num_uml_test_objectives",
        ]
    )

    # COC

    coc = first_numeric_value(
        row,
        [
            "cooccurrence_objective_coverage_score",
        ]
    )

    coc_065 = threshold_value(
        row,
        "cooccurrence_objective_coverage_at",
        0.65
    )

    coc_070 = threshold_value(
        row,
        "cooccurrence_objective_coverage_at",
        0.70
    )

    coc_075 = threshold_value(
        row,
        "cooccurrence_objective_coverage_at",
        0.75
    )

    # Condition-bound objectives

    num_condition_bound = first_numeric_value(
        row,
        [
            "num_condition_bound_objectives",
        ]
    )

    # CBTC

    cbtc = first_numeric_value(
        row,
        [
            "condition_bound_transition_coverage_score",
        ]
    )

    cbtc_065 = threshold_value(
        row,
        "condition_bound_transition_coverage_at",
        0.65
    )

    cbtc_070 = threshold_value(
        row,
        "condition_bound_transition_coverage_at",
        0.70
    )

    cbtc_075 = threshold_value(
        row,
        "condition_bound_transition_coverage_at",
        0.75
    )

    # CBTC applicability

    applicable_col = first_existing_column(
        row,
        [
            "condition_bound_applicable",
        ]
    )

    if applicable_col:

        raw_applicable = str(
            row.iloc[0][
                applicable_col
            ]
        ).strip().lower()

        cbtc_applicable = (
            "Yes"
            if raw_applicable
            in [
                "true",
                "1",
                "yes",
            ]
            else "No"
        )

    else:

        cbtc_applicable = (
            "Yes"
            if (
                pd.notna(
                    num_condition_bound
                )
                and
                num_condition_bound > 0
            )
            else "No"
        )


    # ERPC
    #
    # Existing files may say:
    # oracle_level_coverage
    #
    # Workbook displays same value as ERPC.

    erpc = first_numeric_value(
        row,
        [
            "oracle_level_coverage",
            "expected_result_presence_coverage",
        ]
    )

    num_expected = first_numeric_value(
        row,
        [
            "num_testcases_with_oracle",
            "num_testcases_with_expected_result",
        ]
    )

    num_testcases = first_numeric_value(
        row,
        [
            "num_testcases",
        ]
    )

    result = {
        "Num_UML_Test_Objectives":
            num_objectives,

        "Automated_COC":
            coc,

        "COC_0.65":
            coc_065,

        "COC_0.70":
            coc_070,

        "COC_0.75":
            coc_075,

        "Num_Condition_Bound_Objectives":
            num_condition_bound,

        "CBTC_Applicable":
            cbtc_applicable,

        "Automated_CBTC":
            cbtc,

        "CBTC_0.65":
            cbtc_065,

        "CBTC_0.70":
            cbtc_070,

        "CBTC_0.75":
            cbtc_075,

        "Automated_ERPC":
            erpc,

        "Num_Testcases_With_Expected_Result":
            num_expected,

        "Num_Testcases":
            num_testcases,
    }

    return (
        result,
        additional_path,
        df
    )


# SCAN ALL 580 REPORT/MODEL COMBINATIONS

rows = []

scan_rows = []

semantic_raw_rows = []

additional_raw_frames = []


for (
    llm_folder,
    llm_label
) in LLM_FOLDERS.items():

    llm_dir = os.path.join(
        SEMANTIC_COVERAGE_ROOT,
        llm_folder
    )

    print("\n")
    print("=" * 90)
    print(
        "LLM:",
        llm_folder,
        "->",
        llm_label
    )
    print("=" * 90)

    for (
        vlm_folder,
        vlm_label
    ) in VLM_FOLDERS.items():

        vlm_dir = os.path.join(
            llm_dir,
            vlm_folder
        )

        print(
            "\nVLM:",
            vlm_folder,
            "->",
            vlm_label
        )

        for report_id in EXPECTED_REPORTS:

            report_dir = os.path.join(
                vlm_dir,
                report_id
            )

            # Prepare row even if something is missing.
            #
            # This guarantees the final workbook always has
            # exactly 580 expected combinations.

            output_row = {
                "LLM_Folder":
                    llm_folder,

                "LLM":
                    llm_label,

                "VLM_Folder":
                    vlm_folder,

                "VLM":
                    vlm_label,

                "Report_ID":
                    report_id,

                "Report_Folder_Path":
                    report_dir,
            }

            status = []

            # Report directory

            if not os.path.isdir(
                report_dir
            ):

                status.append(
                    "Report folder missing"
                )

                semantic_result = {}
                additional_result = {}

                semantic_path = None
                additional_path = None

                uml_txt = ""
                testcase_csv = ""

            else:


                # Source artifacts

                (
                    uml_txt,
                    testcase_csv
                ) = find_source_artifacts(
                    report_dir
                )

                # Semantic coverage

                (
                    semantic_result,
                    semantic_path,
                    semantic_raw
                ) = parse_semantic_coverage(
                    report_dir
                )

                if semantic_path is None:

                    status.append(
                        "Semantic result missing"
                    )

                # Store exact semantic raw metric names

                for metric_row in semantic_raw:

                    semantic_raw_rows.append({
                        "LLM_Folder":
                            llm_folder,

                        "LLM":
                            llm_label,

                        "VLM_Folder":
                            vlm_folder,

                        "VLM":
                            vlm_label,

                        "Report_ID":
                            report_id,

                        "Metric":
                            metric_row[
                                "Metric"
                            ],

                        "Value":
                            metric_row[
                                "Value"
                            ],

                        "Source_File":
                            semantic_path,
                    })

                # Additional metrics

                (
                    additional_result,
                    additional_path,
                    additional_raw_df
                ) = parse_additional_metrics(
                    report_dir
                )

                if additional_path is None:

                    status.append(
                        "Additional result missing"
                    )

                # Preserve original additional summary

                if (
                    additional_raw_df is not None
                    and len(
                        additional_raw_df
                    ) > 0
                ):

                    tmp = (
                        additional_raw_df
                        .copy()
                    )

                    tmp.insert(
                        0,
                        "Report_ID",
                        report_id
                    )

                    tmp.insert(
                        0,
                        "VLM",
                        vlm_label
                    )

                    tmp.insert(
                        0,
                        "VLM_Folder",
                        vlm_folder
                    )

                    tmp.insert(
                        0,
                        "LLM",
                        llm_label
                    )

                    tmp.insert(
                        0,
                        "LLM_Folder",
                        llm_folder
                    )

                    tmp[
                        "Source_File"
                    ] = additional_path

                    additional_raw_frames.append(
                        tmp
                    )

            # Add source file paths

            output_row[
                "UML_Description_File"
            ] = uml_txt

            output_row[
                "Generated_Testcase_File"
            ] = testcase_csv

            output_row[
                "Semantic_Result_File"
            ] = (
                semantic_path
                or ""
            )

            output_row[
                "Additional_Result_File"
            ] = (
                additional_path
                or ""
            )

            # Add semantic values

            semantic_defaults = {
                "Semantic_Element_Metric_Name":
                    "",

                "Automated_Element_Coverage":
                    np.nan,

                "Semantic_Flow_Metric_Name":
                    "",

                "Automated_Flow_Coverage":
                    np.nan,

                "Semantic_Conditional_Metric_Name":
                    "",

                "Automated_Conditional_Coverage":
                    np.nan,

                "Semantic_Overall_Metric_Name":
                    "",

                "Automated_ELSC":
                    np.nan,
            }

            semantic_defaults.update(
                semantic_result
            )

            output_row.update(
                semantic_defaults
            )

            # Add additional values

            additional_defaults = {
                "Num_UML_Test_Objectives":
                    np.nan,

                "Automated_COC":
                    np.nan,

                "COC_0.65":
                    np.nan,

                "COC_0.70":
                    np.nan,

                "COC_0.75":
                    np.nan,

                "Num_Condition_Bound_Objectives":
                    np.nan,

                "CBTC_Applicable":
                    "",

                "Automated_CBTC":
                    np.nan,

                "CBTC_0.65":
                    np.nan,

                "CBTC_0.70":
                    np.nan,

                "CBTC_0.75":
                    np.nan,

                "Automated_ERPC":
                    np.nan,

                "Num_Testcases_With_Expected_Result":
                    np.nan,

                "Num_Testcases":
                    np.nan,
            }

            additional_defaults.update(
                additional_result
            )

            output_row.update(
                additional_defaults
            )

            if not status:
                status = ["OK"]

            output_row[
                "Extraction_Status"
            ] = "; ".join(
                status
            )

            rows.append(
                output_row
            )

            scan_rows.append({
                "LLM_Folder":
                    llm_folder,

                "VLM_Folder":
                    vlm_folder,

                "Report_ID":
                    report_id,

                "Status":
                    output_row[
                        "Extraction_Status"
                    ],

                "Semantic_Result_File":
                    output_row[
                        "Semantic_Result_File"
                    ],

                "Additional_Result_File":
                    output_row[
                        "Additional_Result_File"
                    ],
            })


# 14. MASTER DATAFRAME

all_df = pd.DataFrame(
    rows
)

assert len(all_df) == 580, (
    f"Expected 580 rows, got "
    f"{len(all_df)}"
)

# SORT BY EXACT FOLDER STRUCTURE

llm_order = {
    name: i
    for i, name
    in enumerate(
        LLM_FOLDERS.keys()
    )
}

vlm_order = {
    name: i
    for i, name
    in enumerate(
        VLM_FOLDERS.keys()
    )
}


all_df["_llm_order"] = (
    all_df[
        "LLM_Folder"
    ].map(
        llm_order
    )
)

all_df["_vlm_order"] = (
    all_df[
        "VLM_Folder"
    ].map(
        vlm_order
    )
)

all_df["_report_order"] = (
    all_df[
        "Report_ID"
    ].apply(
        report_number
    )
)


all_df = (
    all_df
    .sort_values(
        [
            "_llm_order",
            "_vlm_order",
            "_report_order",
        ]
    )
    .drop(
        columns=[
            "_llm_order",
            "_vlm_order",
            "_report_order",
        ]
    )
    .reset_index(
        drop=True
    )
)

# 16. HUMAN EXPERT INPUT COLUMNS
#
# Experts can enter any value between 0 and 1.
#
# Suggested ordinal scale:
# 0.00
# 0.25
# 0.50
# 0.75
# 1.00
#
# For ERPC, exact manual proportions can also be entered.

human_columns = [
    # Semantic element/entity/concept
    "Expert1_Element_Coverage",
    "Expert2_Element_Coverage",
    "Human_Mean_Element_Coverage",

    # Flow
    "Expert1_Flow_Coverage",
    "Expert2_Flow_Coverage",
    "Human_Mean_Flow_Coverage",

    # Conditional / Validation
    "Expert1_Conditional_Coverage",
    "Expert2_Conditional_Coverage",
    "Human_Mean_Conditional_Coverage",

    # Overall ELSC
    "Expert1_ELSC",
    "Expert2_ELSC",
    "Human_Mean_ELSC",

    # COC
    "Expert1_COC",
    "Expert2_COC",
    "Human_Mean_COC",

    # CBTC
    "Expert1_CBTC",
    "Expert2_CBTC",
    "Human_Mean_CBTC",

    # ERPC
    "Expert1_ERPC",
    "Expert2_ERPC",
    "Human_Mean_ERPC",

    # Notes
    "Expert1_Notes",
    "Expert2_Notes",
]


for col in human_columns:

    all_df[col] = np.nan


# FINAL COLUMN ORDER

FINAL_COLUMNS = [
    # Model/folder identification
    "LLM_Folder",
    "LLM",

    "VLM_Folder",
    "VLM",

    "Report_ID",

    "Extraction_Status",

    # Folder/source traceability

    "Report_Folder_Path",
    "UML_Description_File",
    "Generated_Testcase_File",
    "Semantic_Result_File",
    "Additional_Result_File",

    # SEMANTIC COVERAGE

    "Semantic_Element_Metric_Name",
    "Automated_Element_Coverage",

    "Expert1_Element_Coverage",
    "Expert2_Element_Coverage",
    "Human_Mean_Element_Coverage",

    "Semantic_Flow_Metric_Name",
    "Automated_Flow_Coverage",

    "Expert1_Flow_Coverage",
    "Expert2_Flow_Coverage",
    "Human_Mean_Flow_Coverage",

    "Semantic_Conditional_Metric_Name",
    "Automated_Conditional_Coverage",

    "Expert1_Conditional_Coverage",
    "Expert2_Conditional_Coverage",
    "Human_Mean_Conditional_Coverage",

    "Semantic_Overall_Metric_Name",
    "Automated_ELSC",

    "Expert1_ELSC",
    "Expert2_ELSC",
    "Human_Mean_ELSC",

    # COC

    "Num_UML_Test_Objectives",

    "Automated_COC",

    "COC_0.65",
    "COC_0.70",
    "COC_0.75",

    "Expert1_COC",
    "Expert2_COC",
    "Human_Mean_COC",

    # CBTC

    "Num_Condition_Bound_Objectives",

    "CBTC_Applicable",

    "Automated_CBTC",

    "CBTC_0.65",
    "CBTC_0.70",
    "CBTC_0.75",

    "Expert1_CBTC",
    "Expert2_CBTC",
    "Human_Mean_CBTC",

    # ERPC

    "Num_Testcases",

    "Num_Testcases_With_Expected_Result",

    "Automated_ERPC",

    "Expert1_ERPC",
    "Expert2_ERPC",
    "Human_Mean_ERPC",

    # Human notes

    "Expert1_Notes",
    "Expert2_Notes",
]


all_df = all_df[
    FINAL_COLUMNS
]


# COMBINATION SUMMARY
# Gives one row for every LLM x VLM combination.
# 20 combinations total.


combination_summary = (
    all_df
    .groupby(
        [
            "LLM_Folder",
            "LLM",
            "VLM_Folder",
            "VLM",
        ],
        dropna=False
    )
    .agg(
        Reports=(
            "Report_ID",
            "count"
        ),

        Complete_Rows=(
            "Extraction_Status",
            lambda x: (
                x == "OK"
            ).sum()
        ),

        Mean_Element_Coverage=(
            "Automated_Element_Coverage",
            "mean"
        ),

        Mean_Flow_Coverage=(
            "Automated_Flow_Coverage",
            "mean"
        ),

        Mean_Conditional_Coverage=(
            "Automated_Conditional_Coverage",
            "mean"
        ),

        Mean_ELSC=(
            "Automated_ELSC",
            "mean"
        ),

        Mean_COC=(
            "Automated_COC",
            "mean"
        ),

        Mean_CBTC=(
            "Automated_CBTC",
            "mean"
        ),

        Mean_ERPC=(
            "Automated_ERPC",
            "mean"
        ),
    )
    .reset_index()
)


combination_summary[
    "Incomplete_Rows"
] = (
    combination_summary[
        "Reports"
    ]
    -
    combination_summary[
        "Complete_Rows"
    ]
)

# SCAN / MISSING TABLE


scan_df = pd.DataFrame(
    scan_rows
)

missing_df = scan_df[
    scan_df[
        "Status"
    ] != "OK"
].copy()


# EXACT SEMANTIC RAW METRICS
# This sheet is useful because it preserves the exact
# metric/type name from every semantic summary file.


semantic_raw_df = pd.DataFrame(
    semantic_raw_rows
)

# ADDITIONAL RAW RESULTS
# Keeps ALL original columns from the existing files.

if additional_raw_frames:

    additional_raw_df = pd.concat(
        additional_raw_frames,
        ignore_index=True
    )

else:

    additional_raw_df = pd.DataFrame()

# 22. README SHEET

readme_df = pd.DataFrame(
    [
        [
            "Purpose",
            "Human validation dataset for semantic coverage "
            "and additional coverage metrics."
        ],

        [
            "Source of truth",
            "Each individual Report_X_uml_pages folder is scanned "
            "directly. all_reports_* aggregate folders are NOT used."
        ],

        [
            "Expected rows",
            "580 = 4 LLM folders x 5 VLM folders x 29 reports."
        ],

        [
            "Semantic coverage",
            "Existing semantic_coverage_summary files are read "
            "without recalculation."
        ],

        [
            "Semantic metric types",
            "The workbook preserves whether the original metric "
            "was EntityCoverage/ConceptCoverage and "
            "ConditionalCoverage/ValidationCoverage."
        ],

        [
            "ELSC",
            "Automated_ELSC is the existing SemanticCoverage value."
        ],

        [
            "COC",
            "Automated_COC is the existing "
            "cooccurrence_objective_coverage_score."
        ],

        [
            "CBTC",
            "Automated_CBTC is the existing "
            "condition_bound_transition_coverage_score. "
            "CBTC may be N/A where no condition-bound objective exists."
        ],

        [
            "ERPC",
            "Automated_ERPC is the same existing value stored as "
            "oracle_level_coverage in the original experimental files. "
            "No result is changed."
        ],

        [
            "Thresholds",
            "Existing COC and CBTC values at 0.65, 0.70, and 0.75 "
            "are retained for transparency."
        ],

        [
            "Expert values",
            "Expert 1 and Expert 2 should independently enter values "
            "between 0 and 1 without seeing the automated scores."
        ],

        [
            "Human mean",
            "Human mean columns automatically average the two "
            "expert values."
        ],
    ],
    columns=[
        "Item",
        "Description"
    ]
)

# RUBRIC SHEET

rubric_df = pd.DataFrame(
    [
        [
            "General",
            "",
            "Experts should review the source UML description and "
            "generated test-case specifications independently. "
            "Automated scores should remain hidden during rating."
        ],

        [
            "Recommended rating",
            "0.00",
            "No meaningful correspondence or coverage."
        ],

        [
            "Recommended rating",
            "0.25",
            "Limited correspondence or coverage."
        ],

        [
            "Recommended rating",
            "0.50",
            "Moderate or partial correspondence or coverage."
        ],

        [
            "Recommended rating",
            "0.75",
            "High correspondence or coverage with minor omissions."
        ],

        [
            "Recommended rating",
            "1.00",
            "Complete or nearly complete correspondence or coverage."
        ],

        [
            "Element coverage",
            "0-1",
            "Assess how well the relevant UML entities, concepts, "
            "components, actors, or corresponding elements are "
            "represented in the generated test cases."
        ],

        [
            "Flow coverage",
            "0-1",
            "Assess how well the relevant UML behavioral flows are "
            "represented in the generated test cases."
        ],

        [
            "Conditional coverage",
            "0-1",
            "Assess how well UML conditions, validations, branches, "
            "or corresponding constraints are represented."
        ],

        [
            "ELSC",
            "0-1",
            "Overall judgment of UML-information preservation in the "
            "generated design-based test-case specifications."
        ],

        [
            "COC",
            "0-1",
            "Assess whether semantically related UML elements are "
            "represented together as meaningful test objectives."
        ],

        [
            "CBTC",
            "0-1",
            "Assess whether conditions or guards are represented "
            "together with the corresponding transitions or outcomes. "
            "Leave blank where CBTC_Applicable = No."
        ],

        [
            "ERPC",
            "0-1",
            "Assess the presence of meaningful expected-result "
            "information. This does not assess executable oracle "
            "correctness."
        ],
    ],
    columns=[
        "Metric",
        "Score",
        "Instruction"
    ]
)


# WRITE EXCEL WORKBOOK


with pd.ExcelWriter(
    OUT_XLSX,
    engine="openpyxl"
) as writer:

    readme_df.to_excel(
        writer,
        sheet_name="README",
        index=False
    )

    all_df.to_excel(
        writer,
        sheet_name="Human_Validation_All",
        index=False
    )

    combination_summary.to_excel(
        writer,
        sheet_name="Combination_Summary",
        index=False
    )

    semantic_raw_df.to_excel(
        writer,
        sheet_name="Semantic_Raw_Types",
        index=False
    )

    additional_raw_df.to_excel(
        writer,
        sheet_name="Additional_Raw",
        index=False
    )

    scan_df.to_excel(
        writer,
        sheet_name="Scan_Log",
        index=False
    )

    missing_df.to_excel(
        writer,
        sheet_name="Missing_or_Incomplete",
        index=False
    )

    rubric_df.to_excel(
        writer,
        sheet_name="Rubric",
        index=False
    )

# OPEN WORKBOOK FOR FORMATTING / FORMULAS

wb = load_workbook(
    OUT_XLSX
)

# COLORS

HEADER_FILL = PatternFill(
    fill_type="solid",
    fgColor="1F4E78"
)

HEADER_FONT = Font(
    color="FFFFFF",
    bold=True
)

EXPERT_FILL = PatternFill(
    fill_type="solid",
    fgColor="FFF2CC"
)

MEAN_FILL = PatternFill(
    fill_type="solid",
    fgColor="E2F0D9"
)

AUTO_FILL = PatternFill(
    fill_type="solid",
    fgColor="DDEBF7"
)

SOURCE_FILL = PatternFill(
    fill_type="solid",
    fgColor="E7E6E6"
)

THIN_BORDER = Border(
    bottom=Side(
        style="thin",
        color="D9E2F3"
    )
)


# GENERAL FORMATTING FUNCTION

def format_sheet(
    ws
):

    if ws.max_row < 1:
        return

    for cell in ws[1]:

        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    for row in ws.iter_rows(
        min_row=2
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

            cell.border = (
                THIN_BORDER
            )

    ws.freeze_panes = "A2"

    if (
        ws.max_row > 1
        and ws.max_column > 1
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )

    # Widths

    for col_idx in range(
        1,
        ws.max_column + 1
    ):

        col_letter = (
            get_column_letter(
                col_idx
            )
        )

        max_length = 0

        for cell in ws[
            col_letter
        ]:

            if cell.value is None:
                continue

            max_length = max(
                max_length,
                len(
                    str(
                        cell.value
                    )
                )
            )

        ws.column_dimensions[
            col_letter
        ].width = min(
            max(
                max_length + 2,
                12
            ),
            35
        )


for sheet_name in wb.sheetnames:

    format_sheet(
        wb[sheet_name]
    )


# MAIN HUMAN-VALIDATION SHEET

ws = wb[
    "Human_Validation_All"
]

headers = {
    cell.value: cell.column
    for cell in ws[1]
}


# HUMAN MEAN FORMULAS

mean_pairs = {
    "Human_Mean_Element_Coverage":
        (
            "Expert1_Element_Coverage",
            "Expert2_Element_Coverage",
        ),

    "Human_Mean_Flow_Coverage":
        (
            "Expert1_Flow_Coverage",
            "Expert2_Flow_Coverage",
        ),

    "Human_Mean_Conditional_Coverage":
        (
            "Expert1_Conditional_Coverage",
            "Expert2_Conditional_Coverage",
        ),

    "Human_Mean_ELSC":
        (
            "Expert1_ELSC",
            "Expert2_ELSC",
        ),

    "Human_Mean_COC":
        (
            "Expert1_COC",
            "Expert2_COC",
        ),

    "Human_Mean_CBTC":
        (
            "Expert1_CBTC",
            "Expert2_CBTC",
        ),

    "Human_Mean_ERPC":
        (
            "Expert1_ERPC",
            "Expert2_ERPC",
        ),
}


for (
    mean_col,
    (
        expert1_col,
        expert2_col
    )
) in mean_pairs.items():

    if (
        mean_col not in headers
        or expert1_col not in headers
        or expert2_col not in headers
    ):
        continue

    mean_letter = (
        get_column_letter(
            headers[
                mean_col
            ]
        )
    )

    expert1_letter = (
        get_column_letter(
            headers[
                expert1_col
            ]
        )
    )

    expert2_letter = (
        get_column_letter(
            headers[
                expert2_col
            ]
        )
    )

    for row_num in range(
        2,
        ws.max_row + 1
    ):

        ws[
            f"{mean_letter}{row_num}"
        ] = (
            f'=IF('
            f'COUNT('
            f'{expert1_letter}{row_num}:'
            f'{expert2_letter}{row_num}'
            f')=0,'
            f'"",'
            f'AVERAGE('
            f'{expert1_letter}{row_num}:'
            f'{expert2_letter}{row_num}'
            f')'
            f')'
        )


# HUMAN SCORE VALIDATION
#
# Allows any decimal between 0 and 1.


expert_columns = [
    "Expert1_Element_Coverage",
    "Expert2_Element_Coverage",

    "Expert1_Flow_Coverage",
    "Expert2_Flow_Coverage",

    "Expert1_Conditional_Coverage",
    "Expert2_Conditional_Coverage",

    "Expert1_ELSC",
    "Expert2_ELSC",

    "Expert1_COC",
    "Expert2_COC",

    "Expert1_CBTC",
    "Expert2_CBTC",

    "Expert1_ERPC",
    "Expert2_ERPC",
]


validation = DataValidation(
    type="decimal",
    operator="between",
    formula1="0",
    formula2="1",
    allow_blank=True
)

validation.error = (
    "Enter a value between 0 and 1."
)

validation.errorTitle = (
    "Invalid human score"
)

ws.add_data_validation(
    validation
)


for col_name in expert_columns:

    if col_name not in headers:
        continue

    col_letter = (
        get_column_letter(
            headers[
                col_name
            ]
        )
    )

    validation.add(
        f"{col_letter}2:"
        f"{col_letter}{ws.max_row}"
    )


# HIGHLIGHT COLUMN GROUPS

automatic_columns = [
    "Automated_Element_Coverage",
    "Automated_Flow_Coverage",
    "Automated_Conditional_Coverage",
    "Automated_ELSC",

    "Automated_COC",

    "COC_0.65",
    "COC_0.70",
    "COC_0.75",

    "Automated_CBTC",

    "CBTC_0.65",
    "CBTC_0.70",
    "CBTC_0.75",

    "Automated_ERPC",
]


source_columns = [
    "Report_Folder_Path",
    "UML_Description_File",
    "Generated_Testcase_File",
    "Semantic_Result_File",
    "Additional_Result_File",
]


for col_name in automatic_columns:

    if col_name not in headers:
        continue

    col_idx = headers[
        col_name
    ]

    for row_num in range(
        2,
        ws.max_row + 1
    ):

        ws.cell(
            row=row_num,
            column=col_idx
        ).fill = AUTO_FILL


for col_name in expert_columns:

    if col_name not in headers:
        continue

    col_idx = headers[
        col_name
    ]

    for row_num in range(
        2,
        ws.max_row + 1
    ):

        ws.cell(
            row=row_num,
            column=col_idx
        ).fill = EXPERT_FILL


for col_name in mean_pairs.keys():

    if col_name not in headers:
        continue

    col_idx = headers[
        col_name
    ]

    for row_num in range(
        2,
        ws.max_row + 1
    ):

        ws.cell(
            row=row_num,
            column=col_idx
        ).fill = MEAN_FILL


for col_name in source_columns:

    if col_name not in headers:
        continue

    col_idx = headers[
        col_name
    ]

    for row_num in range(
        2,
        ws.max_row + 1
    ):

        ws.cell(
            row=row_num,
            column=col_idx
        ).fill = SOURCE_FILL


# NUMBER FORMATTING

numeric_score_columns = (
    automatic_columns
    +
    expert_columns
    +
    list(
        mean_pairs.keys()
    )
)


for col_name in numeric_score_columns:

    if col_name not in headers:
        continue

    col_letter = (
        get_column_letter(
            headers[
                col_name
            ]
        )
    )

    for row_num in range(
        2,
        ws.max_row + 1
    ):

        ws[
            f"{col_letter}{row_num}"
        ].number_format = (
            "0.000"
        )


# FREEZE IDENTIFICATION COLUMNS

ws.freeze_panes = "L2"

# SAVE FINAL WORKBOOK

wb.save(
    OUT_XLSX
)

# FINAL DIAGNOSTIC

print("\n")
print("=" * 100)
print("FINAL HUMAN VALIDATION WORKBOOK CREATED")
print("=" * 100)

print("\nOutput file:")
print(
    OUT_XLSX
)

print("\nExpected combinations:")
print(
    4 * 5 * 29
)

print("\nRows created:")
print(
    len(
        all_df
    )
)

print("\nExtraction status:")
print(
    all_df[
        "Extraction_Status"
    ].value_counts(
        dropna=False
    )
)

print("\nNumber of incomplete combinations:")
print(
    len(
        missing_df
    )
)

if len(
    missing_df
) > 0:

    print("\nIncomplete combinations:")

    display(
        missing_df[
            [
                "LLM_Folder",
                "VLM_Folder",
                "Report_ID",
                "Status",
            ]
        ]
    )

print("\nCombination summary:")
display(
    combination_summary
)

print("\nWorkbook sheets:")

for sheet_name in wb.sheetnames:
    print(
        " -",
        sheet_name
    )

print("\nMain sheet for human expert evaluation:")
print(
    "Human_Validation_All"
)

print("\nColor guide:")
print(
    "Blue   = existing automated metric values"
)
print(
    "Yellow = Expert 1 / Expert 2 input"
)
print(
    "Green  = automatically calculated human mean"
)
print(
    "Gray   = source/result file locations"
)

print("\nImportant:")
print(
    "The code reads each Report_X_uml_pages folder directly."
)
print(
    "No all_reports_* aggregate result is used."
)
print(
    "No existing semantic or additional metric is recalculated."
)
print(
    "Automated_ERPC is the same original oracle_level_coverage "
    "value displayed under the revised manuscript terminology."
)

print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


LLM: mistral 7b -> Mistral-7B-Instruct-v0.3

VLM: gemma 4b -> Gemma-3-4B

VLM: gemma 12b -> Gemma-3-12B

VLM: llama -> Llama-3.2-11B-Vision

VLM: llava -> LLaVA-1.6-13B

VLM: qwen -> Qwen2.5-VL-7B


LLM: mistral 14b -> Ministral-3-14B-Instruct-2512

VLM: gemma 4b -> Gemma-3-4B

VLM: gemma 12b -> Gemma-3-12B

VLM: llama -> Llama-3.2-11B-Vision

VLM: llava -> LLaVA-1.6-13B

VLM: qwen -> Qwen2.5-VL-7B


LLM: qwen 7b -> Qwen2.5-7B-Instruct

VLM: gemma 4b -> Gemma-3-4B

VLM: gemma 12b -> Gemma-3-12B

VLM: llama -> Llama-3.2-11B-Vision

VLM: llava -> LLaVA-1.6-13B

VLM: qwen -> Qwen2.5-VL-7B


LLM: qwen 14b -> Qwen2.5-14B-Instruct

VLM: gemma 4b -> Gemma-3-4B

VLM: gemma 12b -> Gemma-3-12B

VLM: llama -> Llama-3.2-11B-Vision

VLM: llava -> LLaVA-1.6-13B

VLM: qwen -> Qwen2.5-VL-7B


FINAL HUMAN VALIDATION WORKBOOK CREATED

Output file:
/content/drive/MyDrive/SEMA

,LLM_Folder,LLM,VLM_Folder,VLM,Reports,Complete_Rows,Mean_Element_Coverage,Mean_Flow_Coverage,Mean_Conditional_Coverage,Mean_ELSC,Mean_COC,Mean_CBTC,Mean_ERPC,Incomplete_Rows
0,mistral 14b,Ministral-3-14B-Instruct-2512,gemma 12b,Gemma-3-12B,29,29,0.748205,0.795416,0.774401,0.772674,0.729367,0.729327,0.962791,0
1,mistral 14b,Ministral-3-14B-Instruct-2512,gemma 4b,Gemma-3-4B,29,29,0.752743,0.789937,0.767420,0.770033,0.735146,0.737365,0.971752,0
2,mistral 14b,Ministral-3-14B-Instruct-2512,llama,Llama-3.2-11B-Vision,29,29,0.745201,0.786306,0.795570,0.775692,0.705038,0.710554,0.972666,0
3,mistral 14b,Ministral-3-14B-Instruct-2512,llava,LLaVA-1.6-13B,29,29,0.743922,0.752108,0.850268,0.782100,0.694520,0.699741,0.990249,0
4,mistral 14b,Ministral-3-14B-Instruct-2512,qwen,Qwen2.5-VL-7B,29,29,0.752651,0.792659,0.798281,0.781197,0.718449,0.715789,0.974168,0
5,mistral 7b,Mistral-7B-Instruct-v0.3,gemma 12b,Gemma-3-12B,29,29,0.746326,0.780316,0.768274,0.764972,0.737491,0.728757,0.926244,0
6,mistral 7b,Mistral-7B-Instruct-v0.3,gemma 4b,Gemma-3-4B,29,29,0.749285,0.786141,0.761153,0.765526,0.744681,0.744365,0.959048,0
7,mistral 7b,Mistral-7B-Instruct-v0.3,llama,Llama-3.2-11B-Vision,29,29,0.758358,0.784725,0.794214,0.779099,0.762874,0.763038,0.919451,0
8,mistral 7b,Mistral-7B-Instruct-v0.3,llava,LLaVA-1.6-13B,29,29,0.751417,0.749024,0.839428,0.779956,0.752003,0.759052,0.985592,0
9,mistral 7b,Mistral-7B-Instruct-v0.3,qwen,Qwen2.5-VL-7B,29,29,0.755294,0.785289,0.792765,0.777783,0.756299,0.746314,0.980432,0



Workbook sheets:
 - README
 - Human_Validation_All
 - Combination_Summary
 - Semantic_Raw_Types
 - Additional_Raw
 - Scan_Log
 - Missing_or_Incomplete
 - Rubric

Main sheet for human expert evaluation:
Human_Validation_All

Color guide:
Blue   = existing automated metric values
Yellow = Expert 1 / Expert 2 input
Green  = automatically calculated human mean
Gray   = source/result file locations

Important:
The code reads each Report_X_uml_pages folder directly.
No all_reports_* aggregate result is used.
No existing semantic or additional metric is recalculated.
Automated_ERPC is the same original oracle_level_coverage value displayed under the revised manuscript terminology.


# FINAL HUMAN EXPERT VALIDATION ANALYSIS
# Metrics:
#   Element / Concept Coverage
#   Flow Coverage
#   Conditional / Validation Coverage
#   ELSC
#   COC
#   CBTC
#   ERPC

# For each metric:
#   - n
#   - Mean Automated Score
#   - Mean Human Score
#   - Spearman's rho
#   - p-value
#   - Quadratically Weighted Cohen's Kappa
#
# Human reference score =
# mean(Human Expert 1, Human Expert 2)
#
# CBTC N/A rows are automatically excluded.


In [ ]:
# IMPORTS

import os
import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score

In [ ]:
# INPUT FILE

# Change only this path if needed.

INPUT_XLSX = (
    "/content/drive/MyDrive/"
    "Human_Expert_Validation_29_Final.xlsx"
)

SHEET_NAME = "Human_Expert_Validation_29"

# OUTPUT FILE

OUTPUT_XLSX = (
    "/content/drive/MyDrive/"
    "Human_Expert_Validation_Final_Statistics.xlsx"
)

# READ DATA

df = pd.read_excel(
    INPUT_XLSX,
    sheet_name=SHEET_NAME
)

print("Rows loaded:", len(df))


# METRICS
# Nothing is grouped into special categories.
# We simply evaluate every score assessed by the experts.


METRICS = {

    "Element/Concept Coverage": {
        "automated": "Automated_Element_Coverage",
        "expert1": "Human_Expert1_Element",
        "expert2": "Human_Expert2_Element",
    },

    "Flow Coverage": {
        "automated": "Automated_Flow_Coverage",
        "expert1": "Human_Expert1_Flow",
        "expert2": "Human_Expert2_Flow",
    },

    "Conditional/Validation Coverage": {
        "automated": "Automated_Conditional_Coverage",
        "expert1": "Human_Expert1_Conditional",
        "expert2": "Human_Expert2_Conditional",
    },

    "ELSC": {
        "automated": "Automated_ELSC",
        "expert1": "Human_Expert1_ELSC",
        "expert2": "Human_Expert2_ELSC",
    },

    "COC": {
        "automated": "Automated_COC",
        "expert1": "Human_Expert1_COC",
        "expert2": "Human_Expert2_COC",
    },

    "CBTC": {
        "automated": "Automated_CBTC",
        "expert1": "Human_Expert1_CBTC",
        "expert2": "Human_Expert2_CBTC",
    },

    "ERPC": {
        "automated": "Automated_ERPC",
        "expert1": "Human_Expert1_ERPC",
        "expert2": "Human_Expert2_ERPC",
    },
}


# CHECK REQUIRED COLUMNS

required_columns = []

for metric_info in METRICS.values():

    required_columns.extend([
        metric_info["automated"],
        metric_info["expert1"],
        metric_info["expert2"],
    ])


missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]


if missing_columns:

    raise ValueError(
        "\nRequired columns are missing:\n"
        + "\n".join(missing_columns)
    )


# CONVERT 0--1 EXPERT SCORE TO ORDINAL CATEGORY
#
# Human scale:
#
# 0.00 -> 0
# 0.25 -> 1
# 0.50 -> 2
# 0.75 -> 3
# 1.00 -> 4
#
# Used only for weighted Cohen's kappa.


def to_ordinal_category(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )

    return np.rint(
        numeric * 4
    ).astype(int)


# RUN ANALYSIS

results = []

validation_rows = []


for metric_name, cols in METRICS.items():

    # Convert values to numeric

    automated = pd.to_numeric(
        df[cols["automated"]],
        errors="coerce"
    )

    expert1 = pd.to_numeric(
        df[cols["expert1"]],
        errors="coerce"
    )

    expert2 = pd.to_numeric(
        df[cols["expert2"]],
        errors="coerce"
    )


    # Human reference = mean of two experts

    human_mean = (
        pd.concat(
            [
                expert1,
                expert2
            ],
            axis=1
        )
        .mean(
            axis=1,
            skipna=False
        )
    )


    # Keep only complete observations
    #
    # This automatically removes:
    #   - missing expert scores
    #   - missing automated scores
    #   - non-applicable CBTC cases


    valid = (
        automated.notna()
        &
        expert1.notna()
        &
        expert2.notna()
        &
        human_mean.notna()
    )


    auto_valid = (
        automated[valid]
        .reset_index(drop=True)
    )

    expert1_valid = (
        expert1[valid]
        .reset_index(drop=True)
    )

    expert2_valid = (
        expert2[valid]
        .reset_index(drop=True)
    )

    human_valid = (
        human_mean[valid]
        .reset_index(drop=True)
    )


    n = len(
        auto_valid
    )



    # SPEARMAN CORRELATION
    #
    # Automated score vs mean human score


    if (
        n >= 3
        and auto_valid.nunique() > 1
        and human_valid.nunique() > 1
    ):

        rho, p_value = spearmanr(
            auto_valid,
            human_valid
        )

    else:

        rho = np.nan
        p_value = np.nan



    # QUADRATICALLY WEIGHTED COHEN'S KAPPA
    #
    # Human Expert 1 vs Human Expert 2

    if n >= 2:

        exp1_category = (
            to_ordinal_category(
                expert1_valid
            )
        )

        exp2_category = (
            to_ordinal_category(
                expert2_valid
            )
        )

        kappa = cohen_kappa_score(
            exp1_category,
            exp2_category,
            weights="quadratic"
        )

    else:

        kappa = np.nan


    # STORE SUMMARY

    results.append({

        "Metric":
            metric_name,

        "n":
            n,

        "Mean_Automated":
            auto_valid.mean(),

        "Mean_Human":
            human_valid.mean(),

        "Spearman_rho":
            rho,

        "p_value":
            p_value,

        "Weighted_Kappa":
            kappa,
    })


    # STORE REPORT-LEVEL VALUES

    for original_index in df.index[valid]:

        validation_rows.append({

            "Metric":
                metric_name,

            "Report_ID":
                df.loc[
                    original_index,
                    "Report_ID"
                ],

            "LLM_Folder":
                df.loc[
                    original_index,
                    "LLM_Folder"
                ],

            "VLM_Folder":
                df.loc[
                    original_index,
                    "VLM_Folder"
                ],

            "Automated_Score":
                automated.loc[
                    original_index
                ],

            "Human_Expert_1":
                expert1.loc[
                    original_index
                ],

            "Human_Expert_2":
                expert2.loc[
                    original_index
                ],

            "Human_Mean":
                human_mean.loc[
                    original_index
                ],
        })


# FINAL SUMMARY TABLE

results_df = pd.DataFrame(
    results
)


# Keep full precision internally
results_full_df = (
    results_df.copy()
)


# Rounded version for manuscript/readability
results_display_df = (
    results_df.copy()
)


for col in [
    "Mean_Automated",
    "Mean_Human",
    "Spearman_rho",
    "Weighted_Kappa",
]:

    results_display_df[col] = (
        results_display_df[col]
        .round(3)
    )


results_display_df[
    "p_value"
] = (
    results_display_df[
        "p_value"
    ]
    .round(4)
)


# DETAILED REPORT-LEVEL DATA

validation_df = pd.DataFrame(
    validation_rows
)


# PRINT FINAL RESULTS

print("\n")
print("=" * 95)
print("FINAL HUMAN EXPERT VALIDATION")
print("=" * 95)

print(
    results_display_df.to_string(
        index=False
    )
)

print("=" * 95)


# SHORT MANUSCRIPT TABLE
#
# This is the compact table I recommend using.

manuscript_df = (
    results_display_df[
        [
            "Metric",
            "n",
            "Mean_Automated",
            "Mean_Human",
            "Spearman_rho",
            "p_value",
            "Weighted_Kappa",
        ]
    ]
    .copy()
)


print("\nMANUSCRIPT TABLE:\n")

display(
    manuscript_df
)


# SAVE TO EXCEL


with pd.ExcelWriter(
    OUTPUT_XLSX,
    engine="openpyxl"
) as writer:

    # Main table
    manuscript_df.to_excel(
        writer,
        sheet_name="Final_Results",
        index=False
    )

    # Full-precision calculations
    results_full_df.to_excel(
        writer,
        sheet_name="Full_Precision",
        index=False
    )

    # Report-level evidence
    validation_df.to_excel(
        writer,
        sheet_name="Validation_Data",
        index=False
    )


# FINAL OUTPUT

print("\nStatistical analysis completed.")

print("\nOutput:")
print(
    OUTPUT_XLSX
)

print(
    "\nUse sheet 'Final_Results' "
    "for the manuscript."
)

print(
    "\n'Validation_Data' is retained only "
    "for traceability."
)

Rows loaded: 29


FINAL HUMAN EXPERT VALIDATION
                         Metric  n  Mean_Automated  Mean_Human  Spearman_rho  p_value  Weighted_Kappa
       Element/Concept Coverage 29           0.751       0.780         0.453   0.0136           0.288
                  Flow Coverage 29           0.782       0.772         0.416   0.0248           0.332
Conditional/Validation Coverage 26           0.788       0.716         0.346   0.0835           0.364
                           ELSC 29           0.774       0.763         0.310   0.1023           0.342
                            COC 29           0.745       0.750         0.291   0.1257           0.466
                           CBTC 26           0.742       0.716         0.252   0.2140           0.364
                           ERPC 29           0.977       0.953         0.964   0.0000           0.000

MANUSCRIPT TABLE:



,Metric,n,Mean_Automated,Mean_Human,Spearman_rho,p_value,Weighted_Kappa
0,Element/Concept Coverage,29,0.751,0.780,0.453,0.0136,0.288
1,Flow Coverage,29,0.782,0.772,0.416,0.0248,0.332
2,Conditional/Validation Coverage,26,0.788,0.716,0.346,0.0835,0.364
3,ELSC,29,0.774,0.763,0.310,0.1023,0.342
4,COC,29,0.745,0.750,0.291,0.1257,0.466
5,CBTC,26,0.742,0.716,0.252,0.2140,0.364
6,ERPC,29,0.977,0.953,0.964,0.0000,0.000



Statistical analysis completed.

Output:
/content/drive/MyDrive/Human_Expert_Validation_Final_Statistics.xlsx

Use sheet 'Final_Results' for the manuscript.

'Validation_Data' is retained only for traceability.
